# LocalGeometry CUDA kernel performance benchmark

This notebook benchmarks the actual performance impact of LocalGeometry on CUDA kernels. It isolates four distinct effects:
1. **Memory bandwidth**: how struct size affects cache efficiency
2. **Broadcast compilation**: how expression complexity affects code generation
3. **Inlining decisions**: how well the compiler optimizes function calls
4. **Overall overhead**: the net impact on real operations

In [1]:
# Enable CUDA and imports
ENV["CLIMACOMMS_DEVICE"] = "CUDA"

using BenchmarkTools
using CUDA
using ClimaComms
using ClimaComms: SingletonCommsContext
using Printf
import ClimaCore
import ClimaCore: Domains, Topologies, Meshes, Spaces, Geometry, Fields, Grids
import ClimaCore.Geometry: LocalGeometry

# Check CUDA availability
CUDA.functional() || error("CUDA not available")

┌ Warning: CUDA runtime library `libcublasLt.so.13` was loaded from a system path, `/usr/local/cuda/lib64/libcublasLt.so.13`.
│ This may cause errors.
│ 
│ If you're running under a profiler, this situation is expected. Otherwise,
│ ensure that your library path environment variable (e.g., `PATH` on Windows
│ or `LD_LIBRARY_PATH` on Linux) does not include CUDA library paths.
│ 
│ In any other case, please file an issue.
└ @ CUDA ~/.julia/packages/CUDA/TPbi4/src/initialization.jl:218
┌ Warning: CUDA runtime library `libnvJitLink.so.13` was loaded from a system path, `/usr/local/cuda/lib64/libnvJitLink.so.13`.
│ This may cause errors.
│ 
│ If you're running under a profiler, this situation is expected. Otherwise,
│ ensure that your library path environment variable (e.g., `PATH` on Windows
│ or `LD_LIBRARY_PATH` on Linux) does not include CUDA library paths.
│ 
│ In any other case, please file an issue.
└ @ CUDA ~/.julia/packages/CUDA/TPbi4/src/initialization.jl:218
┌ Warning: CUDA runt

true

In [2]:
# Load TestUtilities and setup device
# Path from notebooks directory: go up one level to workspace root, then to ClimaCore.jl
testutils_path = abspath(
    joinpath(
        @__DIR__, "..", "ClimaCore.jl", "test", "TestUtilities", "TestUtilities.jl"
    )
)
println("Loading TestUtilities from: $testutils_path")
@isdefined(TU) || include(testutils_path)
import .TestUtilities as TU

device = ClimaComms.CUDADevice()
context = SingletonCommsContext(device)
FT = Float64

println("CUDA Device setup complete. Device: ", CUDA.device())

Loading TestUtilities from: /home/pbachant/calkit/clima-local-geometry/ClimaCore.jl/test/TestUtilities/TestUtilities.jl
CUDA Device setup complete. Device: CuDevice

(0)


In [3]:
# Define helper functions and structs for benchmarking
@inline lg_j_direct(lg) = lg.J
@inline lg_j_stack1(lg) = lg_j_direct(lg)
@inline lg_j_stack2(lg) = lg_j_stack1(lg)
@noinline lg_j_noinline(lg) = lg_j_stack2(lg)

@inline f_from_lg(lg) = lg.J
@noinline f_from_lg_noinline(lg) = lg.J

struct WithLG{LG}
    lg::LG
end

function fd_localgeom_kernel!(out, input, space, nv)
    i = (blockIdx().x - 1) * blockDim().x + threadIdx().x
    if i <= nv
        lg = Geometry.LocalGeometry(space, i, (1, 1, 1))
        @inbounds out[i, 1] = input[i, 1] + lg.J
    end
    return nothing
end

fd_localgeom_kernel! (generic function with 1 method)

In [4]:
# Define simplified geometry structs with varying field counts
struct TwoFieldGeom{FT}
    J::FT
    WJ::FT
end

struct FourFieldGeom{FT}
    J::FT
    WJ::FT
    invJ::FT
    scalar1::FT
end

struct EightFieldGeom{FT}
    J::FT
    WJ::FT
    invJ::FT
    scalar1::FT
    scalar2::FT
    scalar3::FT
    scalar4::FT
    scalar5::FT
end

struct SixteenFieldGeom{FT}
    J::FT
    WJ::FT
    invJ::FT
    scalar1::FT
    scalar2::FT
    scalar3::FT
    scalar4::FT
    scalar5::FT
    scalar6::FT
    scalar7::FT
    scalar8::FT
    scalar9::FT
    scalar10::FT
    scalar11::FT
    scalar12::FT
    scalar13::FT
end

# Testing overhead of nothing fields versus simplified structs
struct TwoFieldWithNothingGeom{FT}
    J::FT
    WJ::FT
    metadata1::Nothing
    metadata2::Nothing
end

struct FourFieldPartiallyNothingGeom{FT}
    J::FT
    WJ::FT
    invJ::Nothing
    extra1::Nothing
end

struct FullGeomWithOptionals{FT}
    J::FT
    WJ::FT
    invJ::FT
    scalar1::FT
    scalar2::FT
    scalar3::FT
end

struct MinimalGeomWithPadding{FT}
    J::FT
    WJ::FT
    padding::NTuple{6,Nothing}
end

@inline function apply_lg_twice_inlined(x, lg)
    return x + lg.J * lg.WJ
end

@noinline function apply_lg_twice_noinline(x, lg)
    return x + lg.J * lg.WJ
end

@inline function apply_lg_three_times_inlined(x, lg)
    return x + lg.J * lg.WJ * lg.invJ
end

@noinline function apply_lg_three_times_noinline(x, lg)
    return x + lg.J * lg.WJ * lg.invJ
end

@inline function complex_lg_computation_inlined(lg::LocalGeometry{FT}) where {FT}
    term1 = lg.J * lg.WJ * lg.invJ
    term2 = lg.J * lg.WJ * lg.invJ
    return term1 + term2
end

@noinline function complex_lg_computation_noinline(lg::LocalGeometry{FT}) where {FT}
    term1 = lg.J * lg.WJ * lg.invJ
    term2 = lg.J * lg.WJ * lg.invJ
    return term1 + term2
end

complex_lg_computation_noinline (generic function with 1 method)

In [5]:
# Create test spaces and fields
space = TU.SpectralElementSpace2D(FT; context=context)
fd_space = TU.ColumnCenterFiniteDifferenceSpace(FT; context=context, zelem=128)

# Create test fields
scalar_field = Fields.Field(FT, space)
result_field = similar(scalar_field)
vector_field = Fields.Field(Geometry.Covariant12Vector{FT}, space)

# Finite-difference space setup
fd_scalar_field = Fields.Field(FT, fd_space)
fd_result_field = similar(fd_scalar_field)
fd_nv = Spaces.nlevels(fd_space)
fd_in = parent(Fields.field_values(fd_scalar_field))
fd_out = parent(Fields.field_values(fd_result_field))
fd_threads = 256
fd_blocks = cld(fd_nv, fd_threads)

println("Created test spaces and fields")
println("  Scalar field size: $(length(scalar_field)) points")
println("  Vector field size: $(length(vector_field)) points")
println("  Finite-difference space: $(fd_nv) levels")

Created test spaces and fields


  Scalar field size: 1 points
  Vector field size: 1 points
  Finite-difference space: 128 levels


In [6]:
# Create geometry fields
local_geom_full = Fields.local_geometry_field(space)

wrapper_lg_field = similar(scalar_field, WithLG{eltype(local_geom_full)})
@. wrapper_lg_field = WithLG(local_geom_full)

# Extract scalar components for comparison
J_field = similar(scalar_field)
@. J_field = local_geom_full.J

# Create fields with simplified geometry structs
two_field_geom = similar(scalar_field, TwoFieldGeom{FT})
@. two_field_geom = TwoFieldGeom(local_geom_full.J, local_geom_full.WJ)

four_field_geom = similar(scalar_field, FourFieldGeom{FT})
@. four_field_geom = FourFieldGeom(local_geom_full.J, local_geom_full.WJ, local_geom_full.invJ, FT(1.0))

eight_field_geom = similar(scalar_field, EightFieldGeom{FT})
@. eight_field_geom = EightFieldGeom(
    local_geom_full.J, local_geom_full.WJ, local_geom_full.invJ,
    FT(1.0), FT(2.0), FT(3.0), FT(4.0), FT(5.0)
)

sixteen_field_geom = similar(scalar_field, SixteenFieldGeom{FT})
@. sixteen_field_geom = SixteenFieldGeom(
    local_geom_full.J, local_geom_full.WJ, local_geom_full.invJ,
    FT(1.0), FT(2.0), FT(3.0), FT(4.0), FT(5.0),
    FT(6.0), FT(7.0), FT(8.0), FT(9.0), FT(10.0),
    FT(11.0), FT(12.0), FT(13.0)
)

# Create fields with nothing-based geometry structs
two_field_with_nothing = similar(scalar_field, TwoFieldWithNothingGeom{FT})
@. two_field_with_nothing = TwoFieldWithNothingGeom(local_geom_full.J, local_geom_full.WJ, nothing, nothing)

four_field_partially_nothing = similar(scalar_field, FourFieldPartiallyNothingGeom{FT})
@. four_field_partially_nothing = FourFieldPartiallyNothingGeom(local_geom_full.J, local_geom_full.WJ, nothing, nothing)

full_geom_with_optionals = similar(scalar_field, FullGeomWithOptionals{FT})
@. full_geom_with_optionals = FullGeomWithOptionals(
    local_geom_full.J, local_geom_full.WJ, local_geom_full.invJ,
    FT(1.0), FT(NaN), FT(NaN),
)

minimal_geom_with_padding = similar(scalar_field, MinimalGeomWithPadding{FT})
make_padding() = (nothing, nothing, nothing, nothing, nothing, nothing)
@. minimal_geom_with_padding = MinimalGeomWithPadding(
    local_geom_full.J, local_geom_full.WJ, make_padding(),
)

simplified_geom = (J=J_field,)

println("Created geometry fields ✓")

Created geometry fields ✓


In [7]:
# Display struct sizes and verify compilation
println("\n" * "="^70)
println("INLINING VERIFICATION")
println("="^70)
println("\nSizeof checks (should help predict inlining threshold):")
println("  LocalGeometry field element:  $(sizeof(eltype(parent(local_geom_full)))) bytes")
println("  TwoFieldGeom:                 $(sizeof(TwoFieldGeom{FT})) bytes")
println("  FourFieldGeom:                $(sizeof(FourFieldGeom{FT})) bytes")
println("  EightFieldGeom:               $(sizeof(EightFieldGeom{FT})) bytes")
println("  SixteenFieldGeom:             $(sizeof(SixteenFieldGeom{FT})) bytes")
println("\nNone field overhead testing:")
println("  TwoFieldWithNothingGeom:      $(sizeof(TwoFieldWithNothingGeom{FT})) bytes")
println("  FourFieldPartiallyNothingGeom: $(sizeof(FourFieldPartiallyNothingGeom{FT})) bytes")
println("  FullGeomWithOptionals:        $(sizeof(FullGeomWithOptionals{FT})) bytes")
println("  MinimalGeomWithPadding:       $(sizeof(MinimalGeomWithPadding{FT})) bytes")
println("\nNote: CUDA typically inlines structs < 128 bytes effectively")
println("Note: sizeof(Nothing) = $(sizeof(Nothing)) bytes")

# Warm up GPU
println("\nWarming up GPU...")
CUDA.synchronize()
@. result_field = scalar_field + 1.0
CUDA.synchronize()
println("GPU warm-up complete ✓")


INLINING VERIFICATION

Sizeof checks (should help predict inlining threshold):
  LocalGeometry field element:  8 bytes
  TwoFieldGeom:                 16 bytes
  FourFieldGeom:                32 bytes
  EightFieldGeom:               64 bytes
  SixteenFieldGeom:             128 bytes

None field overhead testing:
  TwoFieldWithNothingGeom:      16 bytes
  FourFieldPartiallyNothingGeom: 16 bytes
  FullGeomWithOptionals:        48 bytes
  MinimalGeomWithPadding:       16 bytes

Note: CUDA typically inlines structs < 128 bytes effectively
Note: sizeof(Nothing) = 0 bytes

Warming up GPU...


GPU warm-up complete ✓


## Basic geometry access

In [8]:
# SECTION 1: Basic Geometry Access - Define, Run, and Display Results
println("\n" * "="^70)
println("SECTION 1: Basic Geometry Access Patterns")
println("="^70)

section1 = BenchmarkGroup()

section1["1_baseline_simple"] = @benchmarkable begin
    @. $result_field = $scalar_field + 1.0
    CUDA.synchronize()
end

section1["2_full_lg_jacobian"] = @benchmarkable begin
    @. $result_field = $scalar_field + $local_geom_full.J
    CUDA.synchronize()
end

section1["2b_pointwise_lg_j"] = @benchmarkable begin
    @. $result_field = $scalar_field + lg_j_direct($local_geom_full)
    CUDA.synchronize()
end

section1["2c_pointwise_lg_j_stack"] = @benchmarkable begin
    @. $result_field = $scalar_field + lg_j_stack2($local_geom_full)
    CUDA.synchronize()
end

section1["2d_pointwise_lg_j_noinline"] = @benchmarkable begin
    @. $result_field = $scalar_field + lg_j_noinline($local_geom_full)
    CUDA.synchronize()
end

section1["2f_f_x_lg"] = @benchmarkable begin
    @. $result_field = f_from_lg($wrapper_lg_field.lg)
    CUDA.synchronize()
end

section1["2g_lambda_f_x_lg"] = @benchmarkable begin
    $result_field .= (x -> f_from_lg(x.lg)).($wrapper_lg_field)
    CUDA.synchronize()
end

section1["2h_f_x_lg_noinline"] = @benchmarkable begin
    @. $result_field = f_from_lg_noinline($wrapper_lg_field.lg)
    CUDA.synchronize()
end

section1["2i_lambda_f_x_lg_noinline"] = @benchmarkable begin
    $result_field .= (x -> f_from_lg_noinline(x.lg)).($wrapper_lg_field)
    CUDA.synchronize()
end

section1["2e_fd_localgeom_constructor"] = @benchmarkable begin
    CUDA.@sync @cuda threads = $fd_threads blocks = $fd_blocks fd_localgeom_kernel!(
        $fd_out, $fd_in, $fd_space, $fd_nv,
    )
end

section1["3_full_lg_multiple"] = @benchmarkable begin
    @. $result_field = $scalar_field + $local_geom_full.J + $local_geom_full.WJ + $local_geom_full.invJ
    CUDA.synchronize()
end

section1["4_extracted_j"] = @benchmarkable begin
    @. $result_field = $scalar_field + $J_field
    CUDA.synchronize()
end

section1["5_simplified_lg"] = @benchmarkable begin
    @. $result_field = $scalar_field + $simplified_geom.J
    CUDA.synchronize()
end

# Run Section 1
println("\nRunning benchmarks (samples=30)...")
results_s1 = run(section1, verbose=false, samples=30)

# Display Results
println("\nExecution Time (μs, lower is better):")
baseline_time = minimum(results_s1["1_baseline_simple"].times)
baseline_time_μs = baseline_time / 1000

section1_keys = filter(k -> startswith(k, "1_") || startswith(k, "2_") || startswith(k, "3_") || startswith(k, "4_") || startswith(k, "5_"), collect(keys(results_s1)))
for key in sort(section1_keys)
    result = results_s1[key]
    time_μs = minimum(result.times) / 1000
    overhead = 100 * (time_μs - baseline_time_μs) / baseline_time_μs
    overhead_str = overhead >= 0 ? @sprintf("+%.1f", overhead) : @sprintf("%.1f", overhead)
    display_name = replace(key, r"^\d+_|_$" => "")
    display_name = replace(display_name, "_" => " ")
    @printf("  %-40s %10.2f μs  (%6s%% vs baseline)\n", display_name, time_μs, overhead_str)
end


SECTION 1: Basic Geometry Access Patterns

Running benchmarks (samples=30)...



Execution Time (μs, lower is better):


  baseline simple                               15.83 μs  (  +0.0% vs baseline)
  full lg jacobian                              17.24 μs  (  +8.9% vs baseline)


  full lg multiple                              18.64 μs  ( +17.8% vs baseline)
  extracted j                                   16.70 μs  (  +5.5% vs baseline)
  simplified lg                                 16.18 μs  (  +2.2% vs baseline)


## Struct size impact on inlining

In [9]:
# SECTION 2: Struct Size Impact on Inlining - Define, Run, and Display Results
println("\n" * "="^70)
println("SECTION 2: Struct Size Impact on Inlining")
println("="^70)

section2 = BenchmarkGroup()

section2["6_two_field_access"] = @benchmarkable begin
    @. $result_field = $scalar_field + $two_field_geom.J
    CUDA.synchronize()
end

section2["7_four_field_access"] = @benchmarkable begin
    @. $result_field = $scalar_field + $four_field_geom.J
    CUDA.synchronize()
end

section2["8_eight_field_access"] = @benchmarkable begin
    @. $result_field = $scalar_field + $eight_field_geom.J
    CUDA.synchronize()
end

section2["9_sixteen_field_access"] = @benchmarkable begin
    @. $result_field = $scalar_field + $sixteen_field_geom.J
    CUDA.synchronize()
end

# Run Section 2
println("\nRunning benchmarks (samples=30)...")
results_s2 = run(section2, verbose=false, samples=30)

# Display Results
println("\nExecution Time (μs, lower is better):")
for key in sort(collect(keys(results_s2)))
    result = results_s2[key]
    time_μs = minimum(result.times) / 1000
    overhead = 100 * (time_μs - baseline_time_μs) / baseline_time_μs
    overhead_str = overhead >= 0 ? @sprintf("+%.1f", overhead) : @sprintf("%.1f", overhead)
    display_name = replace(key, r"^\d+_" => "")
    display_name = replace(display_name, "_" => " ")
    @printf("  %-40s %10.2f μs  (%6s%% vs baseline)\n", display_name, time_μs, overhead_str)
end


SECTION 2: Struct Size Impact on Inlining

Running benchmarks (samples=30)...

Execution Time (μs, lower is better):


  two field access                              16.72 μs  (  +5.6% vs baseline)
  four field access                             16.79 μs  (  +6.1% vs baseline)
  eight field access                            16.55 μs  (  +4.5% vs baseline)
  sixteen field access                          16.39 μs  (  +3.5% vs baseline)


## `Nothing` field overhead

In [10]:
# SECTION 2B: Nothing Field Overhead - Define, Run, and Display Results
println("\n" * "="^70)
println("SECTION 2B: Nothing Field Overhead Testing")
println("="^70)

section2b = BenchmarkGroup()

section2b["9b_two_field_with_nothing"] = @benchmarkable begin
    @. $result_field = $scalar_field + $two_field_with_nothing.J
    CUDA.synchronize()
end

section2b["9c_four_field_partial_nothing"] = @benchmarkable begin
    @. $result_field = $scalar_field + $four_field_partially_nothing.J
    CUDA.synchronize()
end

section2b["9d_full_geom_with_optionals"] = @benchmarkable begin
    @. $result_field = $scalar_field + $full_geom_with_optionals.J
    CUDA.synchronize()
end

section2b["9e_minimal_with_padding"] = @benchmarkable begin
    @. $result_field = $scalar_field + $minimal_geom_with_padding.J
    CUDA.synchronize()
end

# Run Section 2B
println("\nRunning benchmarks (samples=30)...")
results_s2b = run(section2b, verbose=false, samples=30)

# Display Results
println("\nExecution Time (μs, lower is better):")
for key in sort(collect(keys(results_s2b)))
    result = results_s2b[key]
    time_μs = minimum(result.times) / 1000
    overhead = 100 * (time_μs - baseline_time_μs) / baseline_time_μs
    overhead_str = overhead >= 0 ? @sprintf("+%.1f", overhead) : @sprintf("%.1f", overhead)
    display_name = replace(key, r"^\d+_" => "")
    display_name = replace(display_name, "_" => " ")
    @printf("  %-40s %10.2f μs  (%6s%% vs baseline)\n", display_name, time_μs, overhead_str)
end


SECTION 2B: Nothing Field Overhead Testing

Running benchmarks (samples=30)...

Execution Time (μs, lower is better):


  9b two field with nothing                     16.90 μs  (  +6.8% vs baseline)
  9c four field partial nothing                 16.90 μs  (  +6.8% vs baseline)
  9d full geom with optionals                   16.90 μs  (  +6.8% vs baseline)
  9e minimal with padding                       16.83 μs  (  +6.3% vs baseline)


## Projection operations

In [11]:
# SECTION 3: Projection Operations - Define, Run, and Display Results
println("\n" * "="^70)
println("SECTION 3: Projection Operations")
println("="^70)

section3 = BenchmarkGroup()

# Initialize vector field
@. vector_field = Geometry.Covariant12Vector(scalar_field, scalar_field)
result_vector = similar(vector_field, Geometry.Contravariant12Vector{FT})

section3["10_vector_baseline"] = @benchmarkable begin
    @. $result_vector = Geometry.Contravariant12Vector($scalar_field * 2.0, $scalar_field * 2.0)
    CUDA.synchronize()
end

section3["11_project_full_lg"] = @benchmarkable begin
    @. $result_vector = Geometry.project(Geometry.Contravariant12Axis(), $vector_field, $local_geom_full)
    CUDA.synchronize()
end

section3["12_multiple_scalar_access"] = @benchmarkable begin
    @. $result_field = $scalar_field * ($local_geom_full.J + $local_geom_full.WJ * 0.5)
    CUDA.synchronize()
end

# Run Section 3
println("\nRunning benchmarks (samples=30)...")
results_s3 = run(section3, verbose=false, samples=30)

# Display Results
println("\nExecution Time (μs, lower is better):")
vector_baseline = minimum(results_s3["10_vector_baseline"].times) / 1000
for key in sort(collect(keys(results_s3)))
    result = results_s3[key]
    time_μs = minimum(result.times) / 1000
    overhead = 100 * (time_μs - vector_baseline) / vector_baseline
    overhead_str = overhead >= 0 ? @sprintf("+%.1f", overhead) : @sprintf("%.1f", overhead)
    display_name = replace(key, r"^\d+_" => "")
    display_name = replace(display_name, "_" => " ")
    @printf("  %-40s %10.2f μs  (%6s%% vs vec_baseline)\n", display_name, time_μs, overhead_str)
end


SECTION 3: Projection Operations



Running benchmarks (samples=30)...



Execution Time (μs, lower is better):


  vector baseline                               17.41 μs  (  +0.0% vs vec_baseline)
  project full lg                               16.51 μs  (  -5.2% vs vec_baseline)
  multiple scalar access                        18.01 μs  (  +3.4% vs vec_baseline)


## Fixed expression with varying struct to isolate memory effects

In [12]:
# SECTION 4: Fixed Expression, Varying Struct (isolate memory effects)
println("\n" * "="^70)
println("SECTION 4: Fixed Expression, Varying Struct (isolate memory effects)")
println("="^70)
println("(Same expression @. result_field = scalar_field + struct.J, varying struct size)")

section4 = BenchmarkGroup()

section4["4_1_scalar_with_2field"] = @benchmarkable begin
    @. $result_field = $scalar_field + $two_field_geom.J
    CUDA.synchronize()
end

section4["4_2_scalar_with_4field"] = @benchmarkable begin
    @. $result_field = $scalar_field + $four_field_geom.J
    CUDA.synchronize()
end

section4["4_3_scalar_with_8field"] = @benchmarkable begin
    @. $result_field = $scalar_field + $eight_field_geom.J
    CUDA.synchronize()
end

section4["4_4_scalar_with_16field"] = @benchmarkable begin
    @. $result_field = $scalar_field + $sixteen_field_geom.J
    CUDA.synchronize()
end

section4["4_5_scalar_with_full_lg"] = @benchmarkable begin
    @. $result_field = $scalar_field + $local_geom_full.J
    CUDA.synchronize()
end

# Run Section 4
println("\nRunning benchmarks (samples=30)...")
results_s4 = run(section4, verbose=false, samples=30)

# Display Results
println("\nExecution Time (μs, lower is better):")
baseline_4 = minimum(results_s4["4_1_scalar_with_2field"].times) / 1000
for key in sort(collect(keys(results_s4)))
    result = results_s4[key]
    time_μs = minimum(result.times) / 1000
    overhead = 100 * (time_μs - baseline_4) / baseline_4
    overhead_str = overhead >= 0 ? @sprintf("+%.1f", overhead) : @sprintf("%.1f", overhead)
    display_name = replace(key, r"^\d+_" => "")
    display_name = replace(display_name, "_" => " ")
    struct_info = ""
    if contains(key, "2field")
        struct_info = " (2 fields, 16 bytes)"
    elseif contains(key, "4field")
        struct_info = " (4 fields, 32 bytes)"
    elseif contains(key, "8field")
        struct_info = " (8 fields, 64 bytes)"
    elseif contains(key, "16field")
        struct_info = " (16 fields, 128 bytes)"
    elseif contains(key, "full_lg")
        struct_info = " (Full LocalGeometry)"
    end
    @printf("  %-20s %10.2f μs  (%6s%% vs min)  %s\n", display_name, time_μs, overhead_str, struct_info)
end


SECTION 4: Fixed Expression, Varying Struct (isolate memory effects)
(Same expression @. result_field = scalar_field + struct.J, varying struct size)

Running benchmarks (samples=30)...

Execution Time (μs, lower is better):


  1 scalar with 2field      16.02 μs  (  +0.0% vs min)   (2 fields, 16 bytes)
  2 scalar with 4field      16.51 μs  (  +3.1% vs min)   (4 fields, 32 bytes)


  3 scalar with 8field      16.36 μs  (  +2.1% vs min)   (8 fields, 64 bytes)
  4 scalar with 16field      16.70 μs  (  +4.2% vs min)   (16 fields, 128 bytes)
  5 scalar with full lg      16.16 μs  (  +0.9% vs min)   (Full LocalGeometry)


## Fixed struct with varying expression complexity

In [13]:
# SECTION 5: Fixed Struct, Varying Expression Complexity
println("\n" * "="^70)
println("SECTION 5: Fixed Struct (LocalGeometry), Varying Expression Complexity")
println("="^70)
println("(Same LocalGeometry field, varying number of terms and operations)")

section5 = BenchmarkGroup()

section5["5_1_expr_1term"] = @benchmarkable begin
    @. $result_field = $scalar_field + $local_geom_full.J
    CUDA.synchronize()
end

section5["5_2_expr_2terms"] = @benchmarkable begin
    @. $result_field = $scalar_field + $local_geom_full.J + $local_geom_full.WJ * 0.1
    CUDA.synchronize()
end

section5["5_3_expr_3terms"] = @benchmarkable begin
    @. $result_field = $scalar_field + $local_geom_full.J + $local_geom_full.WJ * 0.1 + $local_geom_full.invJ * 0.01
    CUDA.synchronize()
end

section5["5_4_expr_4terms"] = @benchmarkable begin
    @. $result_field = $scalar_field * ($local_geom_full.J + $local_geom_full.WJ * 0.1 + $local_geom_full.invJ * 0.01 + $scalar_field * 0.001)
    CUDA.synchronize()
end

section5["5_5_expr_6terms"] = @benchmarkable begin
    @. $result_field = ($scalar_field + $local_geom_full.J) *
                       ($local_geom_full.WJ + $local_geom_full.invJ) *
                       (1.0 + $scalar_field * 0.01)
    CUDA.synchronize()
end

# Run Section 5
println("\nRunning benchmarks (samples=30)...")
results_s5 = run(section5, verbose=false, samples=30)

# Display Results
println("\nExecution Time (μs, lower is better):")
baseline_5 = minimum(results_s5["5_1_expr_1term"].times) / 1000
for key in sort(collect(keys(results_s5)))
    result = results_s5[key]
    time_μs = minimum(result.times) / 1000
    overhead = 100 * (time_μs - baseline_5) / baseline_5
    overhead_str = overhead >= 0 ? @sprintf("+%.1f", overhead) : @sprintf("%.1f", overhead)
    display_name = replace(key, r"^\d+_" => "")
    display_name = replace(display_name, "_" => " ")
    complexity_info = ""
    if contains(key, "1term")
        complexity_info = " (1 term)"
    elseif contains(key, "2terms")
        complexity_info = " (2 terms)"
    elseif contains(key, "3terms")
        complexity_info = " (3 terms)"
    elseif contains(key, "4terms")
        complexity_info = " (4 terms, 1 multiply)"
    elseif contains(key, "6terms")
        complexity_info = " (6 terms, nested mults)"
    end
    @printf("  %-20s %10.2f μs  (%6s%% vs baseline)  %s\n", display_name, time_μs, overhead_str, complexity_info)
end


SECTION 5: Fixed Struct (LocalGeometry), Varying Expression Complexity
(Same LocalGeometry field, varying number of terms and operations)

Running benchmarks (samples=30)...



Execution Time (μs, lower is better):


  1 expr 1term              16.04 μs  (  +0.0% vs baseline)   (1 term)
  2 expr 2terms             17.75 μs  ( +10.7% vs baseline)   (2 terms)
  3 expr 3terms             18.70 μs  ( +16.6% vs baseline)   (3 terms)
  4 expr 4terms             19.69 μs  ( +22.8% vs baseline)   (4 terms, 1 multiply)
  5 expr 6terms             20.25 μs  ( +26.2% vs baseline)   (6 terms, nested mults)


## Stencil-like operations

In [14]:
# SECTION 6: Stencil-like Operations (chained LocalGeometry access)
println("\n" * "="^70)
println("SECTION 6: Stencil-like Operations (chained LocalGeometry access)")
println("="^70)
println("(Multiple accesses to LocalGeometry fields in single operation)")

section6 = BenchmarkGroup()

section6["6_1_chained_lg_2accesses_inlined"] = @benchmarkable begin
    @. $result_field = $scalar_field + apply_lg_twice_inlined($scalar_field, $local_geom_full)
    CUDA.synchronize()
end

section6["6_2_chained_lg_2accesses_noinline"] = @benchmarkable begin
    @. $result_field = $scalar_field + apply_lg_twice_noinline($scalar_field, $local_geom_full)
    CUDA.synchronize()
end

section6["6_3_chained_lg_3accesses_inlined"] = @benchmarkable begin
    @. $result_field = $scalar_field + apply_lg_three_times_inlined($scalar_field, $local_geom_full)
    CUDA.synchronize()
end

section6["6_4_chained_lg_3accesses_noinline"] = @benchmarkable begin
    @. $result_field = $scalar_field + apply_lg_three_times_noinline($scalar_field, $local_geom_full)
    CUDA.synchronize()
end

section6["6_5_complex_multi_product_inlined"] = @benchmarkable begin
    @. $result_field = $scalar_field - complex_lg_computation_inlined($local_geom_full)
    CUDA.synchronize()
end

section6["6_6_complex_multi_product_noinline"] = @benchmarkable begin
    @. $result_field = $scalar_field - complex_lg_computation_noinline($local_geom_full)
    CUDA.synchronize()
end

# Run Section 6
println("\nRunning benchmarks (samples=30)...")
results_s6 = run(section6, verbose=false, samples=30)

# Display Results
println("\nExecution Time (μs, lower is better):")
baseline_6 = minimum(results_s1["1_baseline_simple"].times) / 1000
for key in sort(collect(keys(results_s6)))
    result = results_s6[key]
    time_μs = minimum(result.times) / 1000
    overhead = 100 * (time_μs - baseline_6) / baseline_6
    overhead_str = overhead >= 0 ? @sprintf("+%.1f", overhead) : @sprintf("%.1f", overhead)
    display_name = replace(key, r"^\d+_" => "")
    display_name = replace(display_name, "_" => " ")
    @printf("  %-40s %10.2f μs  (%6s%% vs scalar baseline)\n", display_name, time_μs, overhead_str)
end


SECTION 6: Stencil-like Operations (chained LocalGeometry access)
(Multiple accesses to LocalGeometry fields in single operation)

Running benchmarks (samples=30)...



Execution Time (μs, lower is better):


  1 chained lg 2accesses inlined                17.43 μs  ( +10.1% vs scalar baseline)
  2 chained lg 2accesses noinline               19.37 μs  ( +22.4% vs scalar baseline)
  3 chained lg 3accesses inlined                17.73 μs  ( +12.0% vs scalar baseline)
  4 chained lg 3accesses noinline               19.40 μs  ( +22.6% vs scalar baseline)
  5 complex multi product inlined               17.01 μs  (  +7.5% vs scalar baseline)
  6 complex multi product noinline              18.38 μs  ( +16.1% vs scalar baseline)


## Chained stencil operations

In [15]:
# SECTION 7: Chained Stencil Operations (gradient-divergence chains)
println("\n" * "="^70)
println("SECTION 7: Chained Stencil Operations (gradient-divergence chains)")
println("="^70)
println("(Benchmarking common patterns like @. gradᵥ(divᵥ(Y.c.uₕ)))")

section7 = BenchmarkGroup()

# Create operators for finite difference space
import ClimaCore.Operators as Ops

ᶜinterp = Ops.InterpolateF2C()
ᶠinterp = Ops.InterpolateC2F(
    bottom=Ops.Extrapolate(),
    top=Ops.Extrapolate(),
)
ᶜgradᵥ = Ops.GradientF2C()
ᶠgradᵥ = Ops.GradientC2F(
    bottom=Ops.SetGradient(Geometry.WVector(FT(0))),
    top=Ops.SetGradient(Geometry.WVector(FT(0))),
)
ᶠdivᵥ = Ops.DivergenceC2F(
    bottom=Ops.SetValue(Geometry.WVector(FT(0))),
    top=Ops.SetValue(Geometry.WVector(FT(0))),
)
ᶜdivᵥ = Ops.DivergenceF2C()

# Create test fields on finite difference space
ᶜscalar = Fields.Field(FT, fd_space)
ᶠscalar = Fields.Field(FT, Spaces.face_space(fd_space))
ᶜvector = Fields.Field(Geometry.Covariant3Vector{FT}, fd_space)
ᶠvector = Fields.Field(Geometry.Covariant3Vector{FT}, Spaces.face_space(fd_space))
ᶜresult = similar(ᶜscalar)
ᶠresult = similar(ᶠscalar)

# Initialize test fields with constant values
@. ᶜscalar = FT(1.5)
@. ᶠscalar = FT(1.5)
@. ᶜvector = Geometry.Covariant3Vector(ᶜscalar)
@. ᶠvector = Geometry.Covariant3Vector(ᶠscalar)

println("\nCreated finite difference operators and fields")

section7["7_1_baseline_scalar_copy"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜscalar
    CUDA.synchronize()
end

section7["7_2_single_interp_f2c"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜinterp($ᶠscalar)
    CUDA.synchronize()
end

section7["7_3_single_grad_f2c"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜgradᵥ($ᶠscalar).components.data.:1
    CUDA.synchronize()
end

section7["7_4_single_div_f2c"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜdivᵥ($ᶠvector)
    CUDA.synchronize()
end

section7["7_5_grad_then_div"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜdivᵥ($ᶠgradᵥ($ ᶜscalar))
    CUDA.synchronize()
end

section7["7_6_div_then_grad"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜgradᵥ($ᶠdivᵥ($ ᶜvector)).components.data.:1
    CUDA.synchronize()
end

section7["7_7_interp_chain_c2f2c"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜinterp($ᶠinterp($ ᶜscalar))
    CUDA.synchronize()
end

section7["7_8_triple_chain_grad_div_grad"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜgradᵥ($ᶠdivᵥ($ ᶜgradᵥ($ᶠscalar))).components.data.:1
    CUDA.synchronize()
end

# Run Section 7
println("\nRunning benchmarks (samples=30)...")
results_s7 = run(section7, verbose=false, samples=30)

# Display Results
println("\nExecution Time (μs, lower is better):")
baseline_s7 = minimum(results_s7["7_1_baseline_scalar_copy"].times) / 1000
single_interp = minimum(results_s7["7_2_single_interp_f2c"].times) / 1000
single_grad = minimum(results_s7["7_3_single_grad_f2c"].times) / 1000
single_div = minimum(results_s7["7_4_single_div_f2c"].times) / 1000
grad_div = minimum(results_s7["7_5_grad_then_div"].times) / 1000
div_grad = minimum(results_s7["7_6_div_then_grad"].times) / 1000
interp_chain = minimum(results_s7["7_7_interp_chain_c2f2c"].times) / 1000
triple_chain = minimum(results_s7["7_8_triple_chain_grad_div_grad"].times) / 1000

for key in sort(collect(keys(results_s7)))
    result = results_s7[key]
    time_μs = minimum(result.times) / 1000
    overhead = 100 * (time_μs - baseline_s7) / baseline_s7
    overhead_str = overhead >= 0 ? @sprintf("+%.1f", overhead) : @sprintf("%.1f", overhead)
    display_name = replace(key, r"^\d+_" => "")
    display_name = replace(display_name, "_" => " ")
    @printf("  %-40s %10.2f μs  (%6s%% vs baseline)\n", display_name, time_μs, overhead_str)
end

println("\n" * "-"^70)
println("Stencil Chain Analysis:")
println("-"^70)
println("  Single interpolation:             $(@sprintf("%.2f", single_interp)) μs  ($(@sprintf("%.1f", single_interp/baseline_s7))x baseline)")
println("  Single gradient:                  $(@sprintf("%.2f", single_grad)) μs  ($(@sprintf("%.1f", single_grad/baseline_s7))x baseline)")
println("  Single divergence:                $(@sprintf("%.2f", single_div)) μs  ($(@sprintf("%.1f", single_div/baseline_s7))x baseline)")
println("\n  Gradient → Divergence (Laplacian):  $(@sprintf("%.2f", grad_div)) μs  ($(@sprintf("%.1f", grad_div/(single_grad+single_div)))x sum of single ops)")
println("  Divergence → Gradient:              $(@sprintf("%.2f", div_grad)) μs  ($(@sprintf("%.1f", div_grad/(single_grad+single_div)))x sum of single ops)")
println("  Interpolation chain (C→F→C):        $(@sprintf("%.2f", interp_chain)) μs  ($(@sprintf("%.1f", interp_chain/(2*single_interp)))x 2× single interp)")
println("  Triple chain (grad→div→grad):       $(@sprintf("%.2f", triple_chain)) μs  ($(@sprintf("%.1f", triple_chain/(2*single_grad+single_div)))x sum of components)")

if grad_div < 1.2 * (single_grad + single_div)
    println("\n  ✓ EXCELLENT FUSION: Chained operations are well-optimized")
    println("    Compiler successfully fuses stencil operations")
elseif grad_div < 1.5 * (single_grad + single_div)
    println("\n  ⚠️  MODERATE OVERHEAD: Some fusion but not optimal")
    println("    Additional ~$(@sprintf("%.0f", 100*(grad_div/(single_grad+single_div)-1)))% overhead from chaining")
else
    println("\n  ⚠️  HIGH OVERHEAD: Poor fusion of stencil operations")
    println("    Chained operations have significant overhead")
    println("    Consider manual kernel fusion for hot paths")
end


SECTION 7: Chained Stencil Operations (gradient-divergence chains)
(Benchmarking common patterns like @. gradᵥ(divᵥ(Y.c.uₕ)))



Created finite difference operators and fields

Running benchmarks (samples=30)...



Execution Time (μs, lower is better):


  1 baseline scalar copy                        15.45 μs  (  +0.0% vs baseline)
  2 single interp f2c                           17.53 μs  ( +13.5% vs baseline)
  3 single grad f2c                             27.02 μs  ( +74.9% vs baseline)
  4 single div f2c                              17.44 μs  ( +12.9% vs baseline)
  5 grad then div                               20.29 μs  ( +31.3% vs baseline)
  6 div then grad                               29.99 μs  ( +94.1% vs baseline)
  7 interp chain c2f2c                          18.17 μs  ( +17.6% vs baseline)
  8 triple chain grad div grad                  29.94 μs  ( +93.8% vs baseline)

----------------------------------------------------------------------
Stencil Chain Analysis:
----------------------------------------------------------------------
  Single interpolation:             17.53 μs  (1.1x baseline)
  Single gradient:                  27.02 μs  (1.7x baseline)
  Single divergence:                17.44 μs  (1.1x baseline)

  Grad

## Stencil operations with `LocalGeometry` access

In [16]:
# SECTION 7B: Stencil Operations with LocalGeometry Access
println("\n" * "="^70)
println("SECTION 7B: Stencil Operations Combined with LocalGeometry Access")
println("="^70)
println("(Realistic patterns combining stencil ops with geometry field access)")

section7b = BenchmarkGroup()

# Create a field with LocalGeometry data on finite difference space
ᶜlocal_geom_fd = Fields.local_geometry_field(fd_space)
ᶠlocal_geom_fd = Fields.local_geometry_field(Spaces.face_space(fd_space))

section7b["7b_1_baseline_lg_access"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜscalar * $ ᶜlocal_geom_fd.J
    CUDA.synchronize()
end

section7b["7b_2_grad_with_lg"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜgradᵥ($ᶠscalar).components.data.:1 * $ ᶜlocal_geom_fd.J
    CUDA.synchronize()
end

section7b["7b_3_div_with_lg"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜdivᵥ($ᶠvector) * $ ᶜlocal_geom_fd.WJ
    CUDA.synchronize()
end

section7b["7b_4_grad_div_with_lg"] = @benchmarkable begin
    @. $ ᶜresult = $ ᶜdivᵥ($ᶠgradᵥ($ ᶜscalar)) * $ ᶜlocal_geom_fd.J
    CUDA.synchronize()
end

section7b["7b_5_complex_stencil_lg"] = @benchmarkable begin
    @. $ ᶜresult = ($ ᶜdivᵥ($ᶠgradᵥ($ ᶜscalar)) * $ ᶜlocal_geom_fd.J +
                    $ ᶜgradᵥ($ᶠscalar).components.data.:1 * $ ᶜlocal_geom_fd.WJ) * $ ᶜlocal_geom_fd.invJ
    CUDA.synchronize()
end

# Run Section 7B
println("\nRunning benchmarks (samples=30)...")
results_s7b = run(section7b, verbose=false, samples=30)

# Display Results
println("\nExecution Time (μs, lower is better):")
baseline_lg_fd = minimum(results_s7b["7b_1_baseline_lg_access"].times) / 1000
grad_lg = minimum(results_s7b["7b_2_grad_with_lg"].times) / 1000
div_lg = minimum(results_s7b["7b_3_div_with_lg"].times) / 1000
grad_div_lg = minimum(results_s7b["7b_4_grad_div_with_lg"].times) / 1000
complex_lg = minimum(results_s7b["7b_5_complex_stencil_lg"].times) / 1000

for key in sort(collect(keys(results_s7b)))
    result = results_s7b[key]
    time_μs = minimum(result.times) / 1000
    overhead = 100 * (time_μs - baseline_lg_fd) / baseline_lg_fd
    overhead_str = overhead >= 0 ? @sprintf("+%.1f", overhead) : @sprintf("%.1f", overhead)
    display_name = replace(key, r"^\d+_" => "")
    display_name = replace(display_name, "_" => " ")
    @printf("  %-40s %10.2f μs  (%6s%% vs baseline)\n", display_name, time_μs, overhead_str)
end

println("\n" * "-"^70)
println("LocalGeometry Impact in Stencil Operations:")
println("-"^70)

# Compare with Section 7 results
lg_overhead_on_grad = 100 * (grad_lg - single_grad) / single_grad
lg_overhead_on_div = 100 * (div_lg - single_div) / single_div
lg_overhead_on_grad_div = 100 * (grad_div_lg - grad_div) / grad_div

println("  LocalGeometry overhead on gradient:   $(@sprintf("%+.1f", lg_overhead_on_grad))%")
println("  LocalGeometry overhead on divergence: $(@sprintf("%+.1f", lg_overhead_on_div))%")
println("  LocalGeometry overhead on grad∘div:   $(@sprintf("%+.1f", lg_overhead_on_grad_div))%")
println("  Complex expression time:              $(@sprintf("%.2f", complex_lg)) μs ($(@sprintf("%.1f", complex_lg/baseline_lg_fd))x baseline)")

if lg_overhead_on_grad_div < 10
    println("\n  ✓ MINIMAL IMPACT: LocalGeometry adds minimal overhead to stencil ops")
    println("    Compiler effectively optimizes combined operations")
elseif lg_overhead_on_grad_div < 25
    println("\n  ⚠️  MODERATE IMPACT: LocalGeometry adds some overhead")
    println("    Still acceptable for most use cases")
else
    println("\n  ⚠️  SIGNIFICANT IMPACT: LocalGeometry overhead is substantial")
    println("    Consider extracting geometry components before stencil operations")
    println("    Or using more specialized kernels")
end


SECTION 7B: Stencil Operations Combined with LocalGeometry Access
(Realistic patterns combining stencil ops with geometry field access)

Running benchmarks (samples=30)...



Execution Time (μs, lower is better):


  7b 1 baseline lg access                       16.17 μs  (  +0.0% vs baseline)
  7b 2 grad with lg                             27.07 μs  ( +67.4% vs baseline)
  7b 3 div with lg                              18.25 μs  ( +12.9% vs baseline)
  7b 4 grad div with lg                         21.43 μs  ( +32.5% vs baseline)
  7b 5 complex stencil lg                       35.06 μs  (+116.8% vs baseline)

----------------------------------------------------------------------
LocalGeometry Impact in Stencil Operations:
----------------------------------------------------------------------
  LocalGeometry overhead on gradient:   +0.2%
  LocalGeometry overhead on divergence: +4.6%
  LocalGeometry overhead on grad∘div:   +5.6%
  Complex expression time:              35.06 μs (2.2x baseline)

  ✓ MINIMAL IMPACT: LocalGeometry adds minimal overhead to stencil ops
    Compiler effectively optimizes combined operations


## Memory footprint comparison

In [17]:
# Memory Footprint Analysis
println("\n" * "="^70)
println("MEMORY FOOTPRINT COMPARISON")
println("="^70)

lg_size = sizeof(parent(local_geom_full))
j_size = sizeof(parent(J_field))
two_field_size = sizeof(parent(two_field_geom))
four_field_size = sizeof(parent(four_field_geom))
eight_field_size = sizeof(parent(eight_field_geom))
sixteen_field_size = sizeof(parent(sixteen_field_geom))
scalar_size = sizeof(parent(scalar_field))

println("\nData structure size per point:")
println("  Scalar field:                    $(scalar_size / length(scalar_field)) bytes")
println("  TwoFieldGeom:                    $(two_field_size / length(two_field_geom)) bytes")
println("  FourFieldGeom:                   $(four_field_size / length(four_field_geom)) bytes")
println("  EightFieldGeom:                  $(eight_field_size / length(eight_field_geom)) bytes")
println("  SixteenFieldGeom:                $(sixteen_field_size / length(sixteen_field_geom)) bytes")
println("  Full LocalGeometry:              $(lg_size / length(local_geom_full)) bytes")
println("  Extracted J:                     $(j_size / length(J_field)) bytes")

println("\nTotal memory footprint:")
println("  Scalar field:                    $(scalar_size / (1024^2) |> x -> @sprintf("%.3f", x)) MB")
println("  TwoFieldGeom:                    $(two_field_size / (1024^2) |> x -> @sprintf("%.3f", x)) MB ($(two_field_size / scalar_size |> x -> @sprintf("%.1f", x))x scalar)")
println("  FourFieldGeom:                   $(four_field_size / (1024^2) |> x -> @sprintf("%.3f", x)) MB ($(four_field_size / scalar_size |> x -> @sprintf("%.1f", x))x scalar)")
println("  EightFieldGeom:                  $(eight_field_size / (1024^2) |> x -> @sprintf("%.3f", x)) MB ($(eight_field_size / scalar_size |> x -> @sprintf("%.1f", x))x scalar)")
println("  SixteenFieldGeom:                $(sixteen_field_size / (1024^2) |> x -> @sprintf("%.3f", x)) MB ($(sixteen_field_size / scalar_size |> x -> @sprintf("%.1f", x))x scalar)")
println("  Full LocalGeometry:              $(lg_size / (1024^2) |> x -> @sprintf("%.3f", x)) MB ($(lg_size / scalar_size |> x -> @sprintf("%.1f", x))x scalar)")
println("  Extracted J:                     $(j_size / (1024^2) |> x -> @sprintf("%.3f", x)) MB ($(j_size / scalar_size |> x -> @sprintf("%.1f", x))x scalar)")


MEMORY FOOTPRINT COMPARISON

Data structure size per point:
  Scalar field:                    128.0 bytes
  TwoFieldGeom:                    256.0 bytes
  FourFieldGeom:                   512.0 bytes
  EightFieldGeom:                  1024.0 bytes
  SixteenFieldGeom:                2048.0 bytes
  Full LocalGeometry:              2688.0 bytes
  Extracted J:                     128.0 bytes

Total memory footprint:
  Scalar field:                    0.000 MB
  TwoFieldGeom:                    0.000 MB (2.0x scalar)
  FourFieldGeom:                   0.000 MB (4.0x scalar)
  EightFieldGeom:                  0.001 MB (8.0x scalar)


  SixteenFieldGeom:                0.002 MB (16.0x scalar)
  Full LocalGeometry:              0.003 MB (21.0x scalar)
  Extracted J:                     0.000 MB (1.0x scalar)


## Analysis and key findings

In [18]:
# Analysis & Key Findings
println("\n" * "="^70)
println("ANALYSIS & KEY FINDINGS")
println("="^70)

# Extract key metrics from section results
baseline_μs = minimum(results_s1["1_baseline_simple"].times) / 1000
full_lg_μs = minimum(results_s1["2_full_lg_jacobian"].times) / 1000
extracted_μs = minimum(results_s1["4_extracted_j"].times) / 1000

two_field_μs = minimum(results_s2["6_two_field_access"].times) / 1000
four_field_μs = minimum(results_s2["7_four_field_access"].times) / 1000
eight_field_μs = minimum(results_s2["8_eight_field_access"].times) / 1000
sixteen_field_μs = minimum(results_s2["9_sixteen_field_access"].times) / 1000

projection_μs = minimum(results_s3["11_project_full_lg"].times) / 1000
vector_baseline_μs = minimum(results_s3["10_vector_baseline"].times) / 1000

lg_overhead_pct = 100 * (full_lg_μs - baseline_μs) / baseline_μs
projection_overhead_pct = 100 * (projection_μs - vector_baseline_μs) / vector_baseline_μs

# Section 4 analysis: Same expression, varying struct
section4_times = [
    minimum(results_s4["4_1_scalar_with_2field"].times) / 1000,
    minimum(results_s4["4_2_scalar_with_4field"].times) / 1000,
    minimum(results_s4["4_3_scalar_with_8field"].times) / 1000,
    minimum(results_s4["4_4_scalar_with_16field"].times) / 1000,
    minimum(results_s4["4_5_scalar_with_full_lg"].times) / 1000,
]
section4_min = minimum(section4_times)
section4_max = maximum(section4_times)
section4_memory_variance = 100 * (section4_max - section4_min) / section4_min

# Section 5 analysis: Same struct, varying expression
section5_times = [
    minimum(results_s5["5_1_expr_1term"].times) / 1000,
    minimum(results_s5["5_2_expr_2terms"].times) / 1000,
    minimum(results_s5["5_3_expr_3terms"].times) / 1000,
    minimum(results_s5["5_4_expr_4terms"].times) / 1000,
    minimum(results_s5["5_5_expr_6terms"].times) / 1000,
]
section5_min = minimum(section5_times)
section5_max = maximum(section5_times)
section5_compilation_variance = 100 * (section5_max - section5_min) / section5_min

# Section 6 analysis: Stencil operations
section6_times = [
    minimum(results_s6["6_1_chained_lg_2accesses_inlined"].times) / 1000,
    minimum(results_s6["6_2_chained_lg_2accesses_noinline"].times) / 1000,
    minimum(results_s6["6_3_chained_lg_3accesses_inlined"].times) / 1000,
    minimum(results_s6["6_4_chained_lg_3accesses_noinline"].times) / 1000,
    minimum(results_s6["6_5_complex_multi_product_inlined"].times) / 1000,
    minimum(results_s6["6_6_complex_multi_product_noinline"].times) / 1000,
]

# Section 7 analysis: Chained stencil operations
baseline_s7 = minimum(results_s7["7_1_baseline_scalar_copy"].times) / 1000
single_interp = minimum(results_s7["7_2_single_interp_f2c"].times) / 1000
single_grad = minimum(results_s7["7_3_single_grad_f2c"].times) / 1000
single_div = minimum(results_s7["7_4_single_div_f2c"].times) / 1000
grad_div = minimum(results_s7["7_5_grad_then_div"].times) / 1000
div_grad = minimum(results_s7["7_6_div_then_grad"].times) / 1000
interp_chain = minimum(results_s7["7_7_interp_chain_c2f2c"].times) / 1000
triple_chain = minimum(results_s7["7_8_triple_chain_grad_div_grad"].times) / 1000

# Section 7B analysis: Stencil operations with LocalGeometry
baseline_lg_fd = minimum(results_s7b["7b_1_baseline_lg_access"].times) / 1000
grad_lg = minimum(results_s7b["7b_2_grad_with_lg"].times) / 1000
div_lg = minimum(results_s7b["7b_3_div_with_lg"].times) / 1000
grad_div_lg = minimum(results_s7b["7b_4_grad_div_with_lg"].times) / 1000
complex_lg = minimum(results_s7b["7b_5_complex_stencil_lg"].times) / 1000

# Calculate LocalGeometry overhead percentages
lg_overhead_on_grad = 100 * (grad_lg - single_grad) / single_grad
lg_overhead_on_div = 100 * (div_lg - single_div) / single_div
lg_overhead_on_grad_div = 100 * (grad_div_lg - grad_div) / grad_div

println("\n1. BASIC GEOMETRY ACCESS OVERHEAD:")
println("   Full LocalGeometry (J only):      $(@sprintf("%.1f", lg_overhead_pct))%")
println("   Extracted J:                      $(@sprintf("%.1f", 100 * (extracted_μs - baseline_μs) / baseline_μs))%")

println("\n2. STRUCT SIZE IMPACT (accessing single field):")
println("   TwoFieldGeom (16 bytes):          $(@sprintf("%.1f", 100 * (two_field_μs - baseline_μs) / baseline_μs))%")
println("   FourFieldGeom (32 bytes):         $(@sprintf("%.1f", 100 * (four_field_μs - baseline_μs) / baseline_μs))%")
println("   EightFieldGeom (64 bytes):        $(@sprintf("%.1f", 100 * (eight_field_μs - baseline_μs) / baseline_μs))%")
println("   SixteenFieldGeom (128 bytes):     $(@sprintf("%.1f", 100 * (sixteen_field_μs - baseline_μs) / baseline_μs))%")

# Detect inlining threshold
inlining_threshold_detected = false
if abs(sixteen_field_μs - eight_field_μs) > 0.1 * eight_field_μs
    println("\n   ⚠️  INLINING THRESHOLD DETECTED: Performance degrades at ~128 bytes")
    inlining_threshold_detected = true
elseif abs(eight_field_μs - four_field_μs) > 0.1 * four_field_μs
    println("\n   ⚠️  INLINING THRESHOLD DETECTED: Performance degrades at ~64 bytes")
    inlining_threshold_detected = true
else
    println("\n   ✓ All struct sizes show similar performance - good inlining")
end

println("\n3. PROJECTION OPERATIONS OVERHEAD:")
println("   Covariant->Contravariant:         $(@sprintf("%.1f", projection_overhead_pct))%")

println("\n4. MEMORY EFFECT ANALYSIS (Section 4: Same expression, varying struct):")
println("   Memory variance across struct sizes: $(@sprintf("%.1f", section4_memory_variance))%")
if section4_memory_variance > 10
    println("   ⚠️  SIGNIFICANT MEMORY EFFECTS: struct size impacts performance")
    println("      Indicates inefficient memory access patterns or cache effects")
else
    println("   ✓ MINIMAL MEMORY EFFECTS: struct size has negligible impact")
    println("      Good memory access locality")
end

println("\n5. BROADCAST COMPILATION EFFECT (Section 5: Same struct, varying complexity):")
println("   Compilation variance across expression complexity: $(@sprintf("%.1f", section5_compilation_variance))%")
if section5_compilation_variance > 20
    println("   ⚠️  SIGNIFICANT COMPILATION EFFECTS: expression complexity matters")
    println("      More complex broadcasts → larger generated kernels")
    println("      May cause register pressure or reduced occupancy")
elseif section5_compilation_variance > 5
    println("   ⚠️  MODERATE COMPILATION EFFECTS: noticeable with complex expressions")
    println("      Keep broadcast expressions concise when possible")
else
    println("   ✓ MINIMAL COMPILATION EFFECTS: ClimaCore broadcasts compile efficiently")
end

println("\n6. STENCIL OPERATION ANALYSIS (Section 6: Chained operations):")
inlining_gap_2accesses = minimum(results_s6["6_2_chained_lg_2accesses_noinline"].times) / minimum(results_s6["6_1_chained_lg_2accesses_inlined"].times)
inlining_gap_3accesses = minimum(results_s6["6_4_chained_lg_3accesses_noinline"].times) / minimum(results_s6["6_3_chained_lg_3accesses_inlined"].times)
println("   Inlining impact (2 accesses):     $(@sprintf("%.2f", inlining_gap_2accesses))x slower with noinline")
println("   Inlining impact (3 accesses):     $(@sprintf("%.2f", inlining_gap_3accesses))x slower with noinline")
if maximum(section6_times) / minimum(section6_times) > 1.5
    println("   ⚠️  INLINING CRITICAL FOR STENCIL OPS: Large performance gap detected")
    println("      Ensure LocalGeometry access functions are inlined")
    println("      Avoid @noinline annotations on geometry-accessing functions")
else
    println("   ✓ Stencil operations handle LocalGeometry efficiently")
end

println("\n7. CHAINED STENCIL OPERATIONS (Section 7: grad/div chains):")
stencil_fusion_quality = grad_div / (single_grad + single_div)
println("   Gradient-Divergence fusion:       $(@sprintf("%.2f", stencil_fusion_quality))x (ideal: 1.0)")
println("   Divergence-Gradient fusion:       $(@sprintf("%.2f", div_grad / (single_grad + single_div)))x")
println("   Triple chain efficiency:          $(@sprintf("%.2f", triple_chain / (2*single_grad + single_div)))x")

if stencil_fusion_quality < 1.2
    println("   ✓ EXCELLENT FUSION: Stencil chains compile to efficient kernels")
    println("      Broadcast macro successfully fuses operations")
elseif stencil_fusion_quality < 1.5
    println("   ⚠️  MODERATE FUSION: Some overhead in chained stencils")
    println("      Consider profiling specific chains if in hot paths")
else
    println("   ⚠️  POOR FUSION: Significant overhead in stencil chains")
    println("      May benefit from manual kernel fusion")
end

println("\n8. STENCIL + LOCALGEOMETRY INTERACTION (Section 7B):")
println("   LocalGeometry overhead on gradient-divergence: $(@sprintf("%+.1f", lg_overhead_on_grad_div))%")
if lg_overhead_on_grad_div < 10
    println("   ✓ MINIMAL INTERACTION: LocalGeometry and stencils compose well")
elseif lg_overhead_on_grad_div < 25
    println("   ⚠️  MODERATE INTERACTION: Some overhead when combining")
else
    println("   ⚠️  SIGNIFICANT INTERACTION: LocalGeometry impacts stencil performance")
    println("      Consider extracting geometry components before stencil operations")
end


ANALYSIS & KEY FINDINGS

1. BASIC GEOMETRY ACCESS OVERHEAD:
   Full LocalGeometry (J only):      8.9%
   Extracted J:                      5.5%

2. STRUCT SIZE IMPACT (accessing single field):
   TwoFieldGeom (16 bytes):          5.6%
   FourFieldGeom (32 bytes):         6.1%
   EightFieldGeom (64 bytes):        4.5%
   SixteenFieldGeom (128 bytes):     3.5%

   ✓ All struct sizes show similar performance - good inlining



3. PROJECTION OPERATIONS OVERHEAD:
   Covariant->Contravariant:         -5.2%

4. MEMORY EFFECT ANALYSIS (Section 4: Same expression, varying struct):
   Memory variance across struct sizes: 4.2%
   ✓ MINIMAL MEMORY EFFECTS: struct size has negligible impact
      Good memory access locality

5. BROADCAST COMPILATION EFFECT (Section 5: Same struct, varying complexity):
   Compilation variance across expression complexity: 26.2%
   ⚠️  SIGNIFICANT COMPILATION EFFECTS: expression complexity matters
      More complex broadcasts → larger generated kernels
      May cause register pressure or reduced occupancy

6. STENCIL OPERATION ANALYSIS (Section 6: Chained operations):
   Inlining impact (2 accesses):     1.11x slower with noinline
   Inlining impact (3 accesses):     1.09x slower with noinline
   ✓ Stencil operations handle LocalGeometry efficiently

7. CHAINED STENCIL OPERATIONS (Section 7: grad/div chains):
   Gradient-Divergence fusion:       0.46x (ideal: 1.0)
   Divergence-Gradi

## Effect decomposition summary

In [19]:
# Effect Decomposition Summary
println("\n" * "="^70)
println("EFFECT DECOMPOSITION SUMMARY")
println("="^70)

println("\nThis benchmark isolates four distinct effects:")
println("\n1. MEMORY BANDWIDTH (Section 4):")
println("   Impact: $(@sprintf("%.1f", section4_memory_variance))%")
if section4_memory_variance > 10
    println("   → SIGNIFICANT: Memory is a limiting factor")
else
    println("   → MINIMAL: Computation dominates over memory")
end

println("\n2. BROADCAST COMPILATION (Section 5):")
println("   Impact: $(@sprintf("%.1f", section5_compilation_variance))%")
if section5_compilation_variance > 20
    println("   → SIGNIFICANT: Compiler struggles with complex expressions")
else
    println("   → MINIMAL: ClimaCore handles complex expressions well")
end

println("\n3. INLINING DECISIONS (Section 6):")
avg_inlining_impact = (inlining_gap_2accesses + inlining_gap_3accesses) / 2
println("   Impact: $(@sprintf("%.2f", avg_inlining_impact))x")
if avg_inlining_impact > 1.3
    println("   → CRITICAL: Inlining failures cause significant slowdowns")
else
    println("   → MANAGEABLE: Inlining works reasonably well")
end

println("\n4. OVERALL LOCALGEOMETRY OVERHEAD (Section 1 & 2):")
println("   Direct access impact: $(@sprintf("%.1f", lg_overhead_pct))%")
if lg_overhead_pct > 20
    println("   → HIGH: Consider design changes")
elseif lg_overhead_pct > 5
    println("   → MODERATE: Worth optimizing if in hot paths")
else
    println("   → LOW: No major optimization needed")
end


EFFECT DECOMPOSITION SUMMARY

This benchmark isolates four distinct effects:

1. MEMORY BANDWIDTH (Section 4):
   Impact: 4.2%
   → MINIMAL: Computation dominates over memory

2. BROADCAST COMPILATION (Section 5):
   Impact: 26.2%
   → SIGNIFICANT: Compiler struggles with complex expressions

3. INLINING DECISIONS (Section 6):
   Impact: 1.10x
   → MANAGEABLE: Inlining works reasonably well

4. OVERALL LOCALGEOMETRY OVERHEAD (Section 1 & 2):
   Direct access impact: 8.9%
   → MODERATE: Worth optimizing if in hot paths


## Recommendations

In [20]:
# Recommendations
println("\n" * "="^70)
println("RECOMMENDATIONS")
println("="^70)

if lg_overhead_pct > 10 || projection_overhead_pct > 20
    println("\n⚠️  SIGNIFICANT OVERHEAD DETECTED:")
    if lg_overhead_pct > 10
        println("   • Full LocalGeometry access has >10% overhead")
        println("   • Consider: Extract J/WJ at kernel entry")
    end
    if projection_overhead_pct > 20
        println("   • Projection operations have >20% overhead")
        println("   • Consider: Cache projected vectors or simplify metric tensors")
    end
    if inlining_threshold_detected
        println("   • Inlining threshold reached - consider splitting large structs")
    end
    println("\n   ACTION: Refactor hot paths to use simplified geometry types")

elseif lg_overhead_pct > 3 || projection_overhead_pct > 10
    println("\n⚠️  MODERATE OVERHEAD DETECTED:")
    println("   • LocalGeometry overhead: $(@sprintf("%.1f", lg_overhead_pct))%")
    println("   • Projection overhead: $(@sprintf("%.1f", projection_overhead_pct))%")
    println("\n   CONSIDER:")
    println("   1. Extract commonly-used components (J, WJ) at kernel entry")
    println("   2. Profile with nsys to see actual bandwidth/occupancy impact:")
    println("      ./scripts/run-nsys.sh --output=results/nsys/benchmark_lg \\")
    println("          julia --project scripts/benchmark_local_geometry_impact.jl")

else
    println("\n✓ MINIMAL OVERHEAD:")
    println("   • LocalGeometry overhead: $(@sprintf("%.1f", lg_overhead_pct))%")
    println("   • Projection overhead: $(@sprintf("%.1f", projection_overhead_pct))%")
    println("   • Current structure is reasonable for these access patterns")
    println("\n   NOTE: Real physics kernels may show different behavior due to:")
    println("   • Multiple LocalGeometry accesses per kernel")
    println("   • Complex projection operations in tight loops")
    println("   • Recommend profiling actual physics kernels with nsys")
end

println("\n" * "="^70)
println("CODE INSPECTION NOTES")
println("="^70)
println("""
To verify compiler inlining behavior:

1. Check LLVM IR for a simple kernel:
   julia> f(x, lg) = x + lg.J
   julia> @code_llvm f(1.0, first(local_geom_full))

   Look for: Should see direct field access, not function calls

2. Check PTX assembly for CUDA kernels:
   julia> using CUDA
   julia> kernel(x, lg) = (@inbounds x[1] += lg[1].J; nothing)
   julia> @device_code_ptx kernel(CuArray([1.0]), parent(local_geom_full))

   Look for: ld.param instructions (should be minimal for inlined structs)

3. Use nsys to profile actual memory bandwidth:
   ./scripts/run-nsys.sh --output=results/nsys/benchmark_lg

   Then analyze with:
   nsys stats results/nsys/benchmark_lg.nsys-rep

   Look for: Memory bandwidth utilization, kernel occupancy
""")

println("="^70)
println("Benchmark completed successfully ✓")


RECOMMENDATIONS

⚠️  MODERATE OVERHEAD DETECTED:
   • LocalGeometry overhead: 8.9%
   • Projection overhead: -5.2%

   CONSIDER:
   1. Extract commonly-used components (J, WJ) at kernel entry
   2. Profile with nsys to see actual bandwidth/occupancy impact:
      ./scripts/run-nsys.sh --output=results/nsys/benchmark_lg \
          julia --project scripts/benchmark_local_geometry_impact.jl

CODE INSPECTION NOTES
To verify compiler inlining behavior:

1. Check LLVM IR for a simple kernel:
   julia> f(x, lg) = x + lg.J
   julia> @code_llvm f(1.0, first(local_geom_full))

   Look for: Should see direct field access, not function calls

2. Check PTX assembly for CUDA kernels:
   julia> using CUDA
   julia> kernel(x, lg) = (@inbounds x[1] += lg[1].J; nothing)
   julia> @device_code_ptx kernel(CuArray([1.0]), parent(local_geom_full))

   Look for: ld.param instructions (should be minimal for inlined structs)

3. Use nsys to profile actual memory bandwidth:
   ./scripts/run-nsys.sh --output=r

## Code inspection: LLVM IR analysis

Inspect the LLVM intermediate representation to verify compiler inlining behavior for LocalGeometry field access.

In [21]:
# Test 1: Simple kernel accessing LocalGeometry.J field
println("\n" * "="^70)
println("LLVM IR INSPECTION - Simple LocalGeometry Field Access")
println("="^70)

# Define a simple function that accesses lg.J
f_access_j(x, lg) = x + lg.J

# Get the first element of the local geometry field to use as test data
lg_elem = first(parent(local_geom_full))

println("\nFunction: f_access_j(x, lg) = x + lg.J")
println("\nLLVM IR (showing field access pattern):")
println("-"^70)

# Use InteractiveUtils for LLVM inspection
using InteractiveUtils
@code_llvm debuginfo = :none f_access_j(1.0, lg_elem)


LLVM IR INSPECTION - Simple LocalGeometry Field Access


┌ Warning: Performing scalar indexing on task Task (runnable, started) @0x00007f9a08217080.
│ Invocation of getindex resulted in scalar indexing of a GPU array.
│ This is typically caused by calling an iterating implementation of a method.
│ Such implementations *do not* execute on the GPU, but very slowly on the CPU,
│ and therefore should be avoided.
│ 
│ If you want to allow scalar iteration, use `allowscalar` or `@allowscalar`
│ to enable scalar iteration globally or for the operations in question.
└ @ GPUArraysCore ~/.julia/packages/GPUArraysCore/aNaXo/src/GPUArraysCore.jl:145



Function: f_access_j(x, lg) = x + lg.J

LLVM IR (showing field access pattern):
----------------------------------------------------------------------


; Function Signature: f_access_j(Float64, Float64)
; Function Attrs: noreturn
define void @julia_f_access_j_68935(double %"x::Float64", double %"lg::Float64") #0 {
top:
  %jlcallframe1 = alloca [2 x ptr], align 8
  %gcframe2 = alloca [3 x ptr], align 16
  call void @llvm.memset.p0.i64(ptr align 16 %gcframe2, i8 0, i64 24, i1 true)
  %thread_ptr = call ptr asm "movq %fs:0, $0", "=r"() #9
  %tls_ppgcstack = getelementptr i8, ptr %thread_ptr, i64 -8
  %tls_pgcstack = load ptr, ptr %tls_ppgcstack, align 8
  store i64 4, ptr %gcframe2, align 16
  %frame.prev = getelementptr inbounds ptr, ptr %gcframe2, i64 1
  %task.gcstack = load ptr, ptr %tls_pgcstack, align 8
  store ptr %task.gcstack, ptr %frame.prev, align 8
  store ptr %gcframe2, ptr %tls_pgcstack, align 8
  %ptls_field = getelementptr inbounds ptr, ptr %tls_pgcstack, i64 2
  %ptls_load = load ptr, ptr %ptls_field, align 8
  %"box::Float64" = call noalias nonnull align 8 dereferenceable(16) ptr @ijl_gc_pool_alloc_instrumented(ptr %ptl

In [22]:
# Test 2: Function with multiple geometry field accesses
println("\n" * "="^70)
println("LLVM IR INSPECTION - Multiple LocalGeometry Field Accesses")
println("="^70)

# Function accessing multiple fields
f_access_multiple(x, lg) = x + lg.J * lg.WJ + lg.invJ

println("\nFunction: f_access_multiple(x, lg) = x + lg.J * lg.WJ + lg.invJ")
println("\nLLVM IR (showing multiple field access pattern):")
println("-"^70)

# Use InteractiveUtils for LLVM inspection
using InteractiveUtils
@code_llvm debuginfo = :none f_access_multiple(1.0, lg_elem)


LLVM IR INSPECTION - Multiple LocalGeometry Field Accesses

Function: f_access_multiple(x, lg) = x + lg.J * lg.WJ + lg.invJ

LLVM IR (showing multiple field access pattern):
----------------------------------------------------------------------
; Function Signature: f_access_multiple(Float64, Float64)
; Function Attrs: noreturn
define void @julia_f_access_multiple_69019(double %"x::Float64", double %"lg::Float64") #0 {
top:
  %jlcallframe1 = alloca [2 x ptr], align 8
  %gcframe2 = alloca [3 x ptr], align 16
  call void @llvm.memset.p0.i64(ptr align 16 %gcframe2, i8 0, i64 24, i1 true)
  %thread_ptr = call ptr asm "movq %fs:0, $0", "=r"() #9
  %tls_ppgcstack = getelementptr i8, ptr %thread_ptr, i64 -8
  %tls_pgcstack = load ptr, ptr %tls_ppgcstack, align 8
  store i64 4, ptr %gcframe2, align 16
  %frame.prev = getelementptr inbounds ptr, ptr %gcframe2, i64 1
  %task.gcstack = load ptr, ptr %tls_pgcstack, align 8
  store ptr %task.gcstack, ptr %frame.prev, align 8
  store ptr %gcframe2,